# Analysis of Annual FireCCI with Interactive Map Selection

Version: 1.0 (2025-06-05) © R Butzer

This notebook allows you to interactively analyze the annual FireCCI Burned Area from 2000 to 2020 for any region of interest.  

### How to Use:
1. Draw a polygon or rectangle on the map to select a region of interest.
2. Use the slider to select a year from 2000 to 2020.
3. The notebook will display the burned area for the selected year within the drawn region as a colored map layer.
4. To choose a new area, simply delete the current feature by clicking the "Delete" button


### Points of Improvement:
- Add a button to clear the map and reset the selection.
- Implement a feature to download the selected burned area data as a GeoJSON or CSV file.
- chose Countries from dropdown menu

In [13]:
import ee
import geemap
import ipywidgets as widgets
from IPython.display import display
import os
from osgeo import gdal, ogr
import geopandas as gpd


# Earth Engine initialisieren
try:
    ee.Initialize()
except Exception as e:
    ee.Authenticate()
    ee.Initialize()

Grid einlesen

In [11]:
workDir = "A:/_BioGeo/"
gridGPKG =  workDir + "butzerfe/thesis/_00_SHAPEFILES/iberian_GLANCE_EU_150km.gpkg"
#grid_gdf.to_file(r"A:/_BioGeo/butzerfe/thesis/_00_SHAPEFILES/wildE_Grid_GLANCE_EU_150km.shp")
grid_ee_wilde = ee.FeatureCollection("projects/ee-butzerfe/assets/wildE_Grid_GLANCE_EU_150km")
grid_ee_wilde = grid_ee_wilde.filter(ee.Filter.eq('wildE', 1))
#grid_ee_iberian = ee.FeatureCollection("projects/ee-butzerfe/assets/iberian_GLANCE_EU_150km")
ogr.UseExceptions()
drvMemV = ogr.GetDriverByName('Memory')
grid = drvMemV.CopyDataSource(ogr.Open(gridGPKG),'')
gridLYR = grid.GetLayer()

In [14]:
grid_ee = grid_ee_wilde

In [ ]:
# show info for ee FeatureCollection


In [26]:
# FireCCI Collection
firecci = ee.ImageCollection("ESA/CCI/FireCCI/5_1")

# Jahre definieren
start_year = 2001
end_year = 2021
years = list(range(start_year, end_year + 1))


##############################
## 1. burn count aggregation

# Für jedes Jahr: Maske, wo ein Brand stattgefunden hat (BurnDate > 0)
def burned_mask(year):
    yearly = firecci.filterDate(f'{year}-01-01', f'{year}-12-31').select('BurnDate')
    # Prüfen, ob Bilder vorhanden sind
    def _mask(imgcol):
        size = imgcol.size()
        return ee.Algorithms.If(
            size.gt(0),
            imgcol.max().gt(0).rename('burned'),
            ee.Image(0).rename('burned').clip(grid_ee)
        )
    return ee.Image(_mask(yearly))

burned_images = [burned_mask(year) for year in years]
burned_stack = ee.ImageCollection(burned_images).sum().clip(grid_ee)
burned_stack_masked = burned_stack.updateMask(burned_stack.gt(0))

scale = 3000  # 3x3 km

# Standardprojektion setzen (z.B. EPSG:4326 mit 3000m Pixelgröße)
burned_proj = burned_stack_masked.setDefaultProjection('EPSG:4326', None, scale)

# Dann aggregieren
burned_agg = burned_proj.reduceResolution(
    reducer=ee.Reducer.sum(), maxPixels=1024
).reproject(crs='EPSG:4326', scale=scale)

##################################
## 2. Burn date pro Jahr

# Für jedes Jahr: Maske, wo ein Brand stattgefunden hat (BurnDate > 0)
def burned_mask(year):
    yearly = firecci.filterDate(f'{year}-01-01', f'{year}-12-31').select('BurnDate')
    def _mask(imgcol):
        size = imgcol.size()
        return ee.Algorithms.If(
            size.gt(0),
            imgcol.max().gt(0).rename(str(year)),
            ee.Image(0).rename(str(year)).clip(grid_ee)
        )
    return ee.Image(_mask(yearly))

burned_images = [burned_mask(year) for year in years]

# Multi-Band-Image erstellen (jedes Band = ein Jahr)
burned_multiband = ee.Image.cat(burned_images).clip(grid_ee)

# Auf 3x3 km aggregieren (Mittelwert pro Zelle = Anteil gebrannter Pixel)
scale = 3000  # 3x3 km
burned_multiband_proj = burned_multiband.setDefaultProjection('EPSG:4326', None, scale)
burned_multiband_agg = burned_multiband_proj.reduceResolution(
    reducer=ee.Reducer.mean(), maxPixels=1024
).reproject(crs='EPSG:4326', scale=scale)

# Nach Aggregation binär machen (0 oder 1)
burned_multiband_bin = burned_multiband_agg.gt(0)


# burned agg zu burned_multiband_bin zufügen
burned_multiband_bin = burned_multiband_bin.addBands(burned_agg.rename('burned_frequency'))


In [ ]:
print(burned_multiband_bin.bandNames().getInfo())
print('Anzahl Bänder:', burned_multiband_bin.bandNames().size().getInfo())
import pprint
pprint.pprint(burned_multiband_bin.getInfo())

In [16]:
############################################################################
# visualisierung
Map = geemap.Map(basemap='SATELLITE')


Map.addLayer(grid_ee, {}, "GLANCE EU 150km")

Map.centerObject(grid_ee, zoom=6)

year = 2001
vis_bin = {'min': 0, 'max': 1, 'palette': ['white', 'red']}
Map.addLayer(
    burned_multiband_bin.select(str(year)),
    vis_bin,
    f'Burned {year} (binär, 3x3km)'
)

# Visualisierung der Frequenz
vis_freq = {'min': 1, 'max': len(years), 'palette': ['yellow', 'orange', 'red', 'darkred']}
Map.addLayer(
    burned_multiband_bin.select('burned_frequency'),
    vis_freq,
    'Burned Frequency (3x3km)'
)

display(Map)


Map(center=[52.698505338662294, 24.927090282479863], controls=(WidgetControl(options=['position', 'transparent…

In [ ]:
# Datentyp anpassen (z.B. Int16 für binäre und Summenbänder)
burned_multiband_bin = burned_multiband_bin.toInt16()


In [25]:
import ee
import os
import time
from tqdm import tqdm
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import getpass

#workDir = r"\\141.20.141.12\SAN_BioGeo\_HiWi\Ruben\thesis"
workDir = r"\\141.20.140.57\DAS_gsz1\_Biogeo\butzerfe\thesis"
DL_folder = workDir + "_Runs/02_Fire/"
if not os.path.exists(DL_folder):
    os.makedirs(DL_folder)

user = getpass.getuser()
if user == "butzerfe":
    drive_folder        = "GEE"
    drive_folder_url    = "10IeS4uYZ8gmy1Nbfh_kQme9Scfp6txg0"
    GoogleAuth.DEFAULT_SETTINGS['client_config_file'] = "L:/_HiWi/Ruben/wildE/client_secrets_ruben_geopy.json"

 # PyDrive Authentifizierung
    gauth = GoogleAuth()
    gauth.LocalWebserverAuth()
    drive = GoogleDrive(gauth)

# Export als GeoTIFF zu Google Drive
task = ee.batch.Export.image.toDrive(
    image=burned_multiband_bin,
    description='FireCCI_Burned_3km_2001-2021',
    folder='GEE',  # Name des Zielordners in Google Drive
    fileNamePrefix='FireCCI_Burned_3km_2001-2021',
    region=grid_ee.geometry(),
    scale=3000,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task.start()
print("Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.")


# Warten bis Export abgeschlossen ist
while task.active():
    print("Export läuft...")
    time.sleep(10)

if task.status()['state'] == 'COMPLETED':
    print("Export erfolgreich abgeschlossen! Starte Download von Google Drive...")
else:
    print("Export fehlgeschlagen:", task.status())

# Download processed tiles from GDrive
drive_list = drive.ListFile({'q': f"'{drive_folder_url}' in parents and trashed=false"}).GetList()
for file in tqdm(drive_list):
    file_id = drive.CreateFile({'id': file['id']})
    fname = file["title"]  # Dateiname auf Google Drive
    outName = os.path.join(DL_folder, fname)
    file_id.GetContentFile(outName)
    file_id.Delete()
    print(f"{fname} heruntergeladen und aus Google Drive gelöscht.")

print("Alle verfügbaren Dateien wurden heruntergeladen.")

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=579762042005-bktd78digk65s9sht4hho28rq5s30apo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code

Authentication successful.
Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export erfolgreich abgeschlossen! Starte Download von Google Drive...


100%|██████████| 1/1 [00:03<00:00,  3.20s/it]

FireCCI_Burned_3km_2001-2021.tif heruntergeladen und aus Google Drive gelöscht.
Alle verfügbaren Dateien wurden heruntergeladen.


In [24]:
import ee
import os
import time
from tqdm import tqdm
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive
import getpass

workDir = r"\\141.20.141.12\SAN_BioGeo\_HiWi\Ruben\thesis"
#workDir = r"\\141.20.140.57\DAS_Morpork\_Biogeo\butzerfe\thesis"
DL_folder = workDir + "_Runs/02_Fire/"
if not os.path.exists(DL_folder):
    os.makedirs(DL_folder)

user = getpass.getuser()
if user == "butzerfe":
    drive_folder        = "GEE"
    drive_folder_url    = "10IeS4uYZ8gmy1Nbfh_kQme9Scfp6txg0"
    GoogleAuth.DEFAULT_SETTINGS['client_config_file'] = "L:/_HiWi/Ruben/wildE/client_secrets_ruben_geopy.json"

 # PyDrive Authentifizierung
    gauth = GoogleAuth()
    gauth.LocalWebserverAuth()
    drive = GoogleDrive(gauth)

# Export als GeoTIFF zu Google Drive
task = ee.batch.Export.image.toDrive(
    image=burned_multiband_bin,
    description='FireCCI_Burned_3km_2001-2021',
    folder='GEE',  # Name des Zielordners in Google Drive
    fileNamePrefix='FireCCI_Burned_3km_2001-2021',
    region=grid_ee.geometry(),
    scale=3000,
    maxPixels=1e13,
    fileFormat='GeoTIFF'
)
task.start()
print("Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.")


# Warten bis Export abgeschlossen ist
while task.active():
    print("Export läuft...")
    time.sleep(10)

if task.status()['state'] == 'COMPLETED':
    print("Export erfolgreich abgeschlossen! Starte Download von Google Drive...")
else:
    print("Export fehlgeschlagen:", task.status())

# Download processed tiles from GDrive
drive_list = drive.ListFile({'q': f"'{drive_folder_url}' in parents and trashed=false"}).GetList()
for file in tqdm(drive_list):
    file_id = drive.CreateFile({'id': file['id']})
    fname = file["title"]  # Dateiname auf Google Drive
    outName = os.path.join(DL_folder, fname)
    file_id.GetContentFile(outName)
    file_id.Delete()
    print(f"{fname} heruntergeladen und aus Google Drive gelöscht.")

print("Alle verfügbaren Dateien wurden heruntergeladen.")

Your browser has been opened to visit:

    https://accounts.google.com/o/oauth2/auth?client_id=579762042005-bktd78digk65s9sht4hho28rq5s30apo.apps.googleusercontent.com&redirect_uri=http%3A%2F%2Flocalhost%3A8080%2F&scope=https%3A%2F%2Fwww.googleapis.com%2Fauth%2Fdrive&access_type=offline&response_type=code

Authentication successful.
Export gestartet. Überprüfe den Task-Status in der Earth Engine-Taskliste.
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export läuft...
Export erfolgreich abgeschlossen! Starte Down

100%|██████████| 1/1 [00:03<00:00,  3.36s/it]

FireCCI_Burned_3km_2001-2021.tif heruntergeladen und aus Google Drive gelöscht.
Alle verfügbaren Dateien wurden heruntergeladen.


andere visualisierungen

In [ ]:
# # FeatureCollection (dein Grid)
# grid_ee = ee.FeatureCollection("projects/ee-butzerfe/assets/iberian_GLANCE_EU_150km")

# # FireCCI Collection
# firecci = ee.ImageCollection("ESA/CCI/FireCCI/5_1")

# # Karte initialisieren
# Map = geemap.Map(basemap='SATELLITE')
# Map.addLayer(grid_ee, {}, "Iberian GLANCE EU 150km")
# Map.centerObject(grid_ee, zoom=6)

# # Jahre und Visualisierung
# fire_years = list(range(2001, 2022))
# baVis = {
#     'min': 1,
#     'max': 366,
#     'palette': [
#         'ff0000', 'fd4100', 'fb8200', 'f9c400', 'f2ff00', 'b6ff05',
#         '7aff0a', '3eff0f', '02ff15', '00ff55', '00ff99', '00ffdd',
#         '00ddff', '0098ff', '0052ff', '0210ff', '3a0dfb', '7209f6',
#         'a905f1', 'e102ed', 'ff00cc', 'ff0089', 'ff0047', 'ff0004'
#     ]
# }
# fire_slider = widgets.SelectionSlider(
#     options=fire_years,
#     value=fire_years[0],
#     description='Fire Jahr',
#     continuous_update=False
# )

# def update_fire(change):
#     # Nur Basemap und Grid-Layer behalten
#     Map.layers = [Map.layers[0], Map.layers[1]]
#     year = fire_slider.value
#     yearly = firecci.filterDate(f'{year}-01-01', f'{year}-12-31').select('BurnDate')
#     if yearly.size().getInfo() > 0:
#         yearly_max = yearly.max().clip(grid_ee)
#         Map.addLayer(yearly_max, baVis, f'FireCCI {year}')

# fire_slider.observe(update_fire, names='value')

# # Initiales Jahr anzeigen
# update_fire(None)

# ui = widgets.VBox([fire_slider, Map])
# display(ui)